# 21 cm signal loss under foreground-mode filtering

The eigenmode analysis shows that the simulated beam-weighted foregrounds occupy a low-dimensional spectral subspace. On its own that says nothing about whether the cosmological signal survives the same filter, so here we push an ensemble of global 21 cm models through the *identical* projection onto the leading $N$ foreground modes and read the retained signal off the same axes as the residuals.

Panels (a1)/(a2) show a legibility subsample of the surviving ensemble before and after filtering $N$ modes (panel (b) draws every model), each coloured by the RMS it retains at $N$ -- the same quantity plotted in panel (b), on the same log scale, so a curve's colour and its height in (b) agree. What survives is small but still structured; note it is band-edge ringing from projecting onto a truncated smooth basis, not a residual trough, so retained RMS is not retained signal *shape*. Panel (b) adds the foreground residual in black: colour marks the subject, greyscale the floor it is measured against. It supersedes `foreground_svd_residual.pdf` -- the black curve is the same one, now never shown without the signal beside it.

**Scope.** This figure makes the smallest claim that supports the design argument: the beam-weighted foregrounds are spectrally low-dimensional, and a signal survives projecting that subspace out. Nothing else is folded in -- no instrumental systematics, no noise, and no knowledge of the beam or the sky. The $+1$ m antenna-position systematic was previously drawn here as a second floor and has been removed: it is not defined until the forward-modelling section, and `horizon_shift.ipynb` already shows it against this same retained-signal benchmark, where folding it in costs one additional mode. Read this as a statement about spectral shape overlap, not as the analysis EIGSEP will run -- the planned analysis is a joint differentiable forward-model fit in which beam chromaticity is modelled rather than filtered.

**Retention is a continuum and no single summary statistic predicts it**, which is why the models are coloured continuously rather than binned into classes. It is set by how much of a model's spectral shape lies in the leading foreground modes. Separating shape from amplitude over the ensemble (Spearman): trough width correlates with the retained *fraction* at -0.59 -- narrower troughs keep proportionally more -- while the *absolute* retained RMS correlates with amplitude only moderately (depth, +0.53). Neither predicts retention alone: the three classes' median depths rise with retained RMS (77, 92, 156 mK for < 2 mK, 2-4.5 mK and > 4.5 mK) but even at matched depth (80-160 mK) they still separate by width -- median trough widths of 36, 20 and 18 MHz respectively. The summary cell bins the distribution at 2 and 4.5 mK for the caption; those bins are a reporting convenience, not populations.

The colour scale is logarithmic. Retained RMS spans 1.54 decades (0.60-21.0 mK), and a linear norm would put 44% of the models in the bottom 10% of the colour range, against 2% for log.

Curve opacity ramps with retained RMS rather than being uniform: the high-retention tail is ~2% of the ensemble drawn under ~1700 other curves, and at any single alpha it reads as empty. Draw order alone does not fix this -- sorting by retention just chooses which end of the colour scale to bury -- so the order is a seeded random permutation and the ramp does the work. Its cost is that opacity echoes the quantity colour already encodes, making the top of the scale look more populated than it is; the ramp is cubic to keep the boost confined to the extreme tail, and the true tail fraction (5.5% of models above 10 mK) is quoted rather than eyeballed.

$N = 9$ is the smallest $N$ at which the foreground residual falls below the median retained signal *and stays below* for every larger $N$ (1.82 vs 3.01 mK; at $N = 8$ it is still above, 7.75 vs 4.94 mK). The 'stays below' clause matters because both curves fall with $N$ and cross more than once, so a first-crossing rule can select an $N$ the floor later climbs back above; `horizon_shift.ipynb` applies the same rule to the position systematic. $N = 9$ is the floor of the range rather than a requirement -- every systematic folded in later can only push it up.

**Limitations, to be stated wherever this result is used.** The modes come from a single simulated sky (GSM16) and beam, with no noise and no receiver systematics; in practice the basis would be estimated from data that already contain the signal, which costs additional signal loss not captured here. Filtering is a hard projection, whereas a joint signal-plus-foreground fit would recover some of what is removed. Signal loss is severe in absolute terms, and whether the retained amplitude is detectable is set by thermal noise and integration time, which this calculation does not model. This is a statement about spectral subspace overlap, not a sensitivity forecast.

Ensemble: 1769 of 4096 Zeus21 models (Munoz 2023a, arXiv:2302.08506, with Pop III and Lyman-Werner feedback, Cruz+2024, arXiv:2407.18294) survive a posterior reionization cut requiring xHI below threshold both at the McGreer+2015 reference redshift and at the top of the observed band (z = 4.6816, 250 MHz). The npz carries its own regeneration recipe (`provenance`, `generator_source`, `env_lock` keys); see `docs/superpowers/specs/2026-08-19-zeus21-model-ensemble-design.md`.

The ensemble runs to $z = 4.65$ (251.4 MHz), below Zeus21's advertised $z = 5$-35 validity range. This is deliberate, so the 250 MHz band edge is covered by computed values rather than extrapolation; zero-padding above Zeus21's native top of range ($z = 5$, 236.7 MHz) was considered and rejected instead, because late-reionization models still carry up to ~14 mK of signal at 237 MHz and the resulting step discontinuity would survive a smooth-mode filter and inflate the retained-RMS statistic reported here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm

In [ ]:
d = np.load("signal_loss.npz", allow_pickle=True)
freqs = d["freqs_MHz"]           # (n_f,) MHz
Vh = d["Vh"]                     # (n_f, n_f) foreground spectral modes
s_fg = d["s_fg"]                 # (n_f,) singular values of the T_ant waterfall
n_time = int(d["n_time"])        # LST samples in that waterfall
T21 = d["T21_models"]            # (n_model, n_f) global-signal ensemble [K]
cls = d["cls"]                   # (n_model,) retained-RMS bin, for the summary
class_labels = [str(x) for x in d["class_labels"]]
draw_idx = d["draw_idx"]         # (n_draw,) curves the figure draws
n_f = freqs.size
N_SHOW = 18                      # x-axis extent
N_ANCHOR = int(d["n_anchor"])    # modes filtered at the quoted operating point
# Opacity ramps with retained RMS, from ALPHA_LO at the bottom of the colour
# scale to ALPHA_HI at the top. At a single uniform alpha the sparse
# high-retention tail is invisible: it is ~2% of the ensemble, drawn under
# ~1700 other curves. The ramp, not the draw order, is what makes it legible
# -- rendering the two effects separately shows the ramp alone reproduces the
# result. The cost is that opacity now echoes the quantity colour already
# carries, which exaggerates how much of the ensemble sits at the top end;
# RAMP_POW > 1 keeps the boost confined to the genuinely extreme tail.
ALPHA_LO, ALPHA_HI, RAMP_POW = 0.07, 0.63, 3.0   # panel (b)
ALL_ALPHA_LO, ALL_ALPHA_HI = 0.07, 0.55          # panels (a1)/(a2)
Z_SEED = 20260820                # seeds the draw order; see PLOT_SRC
CONT_CMAP = "plasma"             # colour = retained RMS, continuous
print(f"{T21.shape[0]} 21 cm models on {n_f} channels, "
      f"{freqs[0]:.0f}-{freqs[-1]:.0f} MHz")

In [ ]:
n_modes = np.arange(N_SHOW + 1)

# Foreground residual after filtering the leading N modes [K] -- the curve
# that used to be foreground_svd_residual.pdf.
tail = np.concatenate([np.cumsum(s_fg[::-1] ** 2)[::-1], [0.0]])
fg_resid = np.sqrt(tail / (n_time * n_f))[: N_SHOW + 1]


def filt_rms(x):
    """RMS over frequency after filtering the leading N modes, per row."""
    c = np.atleast_2d(x) @ Vh.T
    return np.array([np.sqrt(np.sum(c[:, N:] ** 2, axis=1) / n_f)
                     for N in n_modes])                    # (N_SHOW+1, n_row)


def filtered(x, N):
    """The part of x left after projecting out the leading N modes."""
    c = np.atleast_2d(x) @ Vh.T
    return (c[:, N:] @ Vh[N:]).reshape(np.shape(x))


t21_resid = filt_rms(T21)                                  # (N_SHOW+1, n_model)
t21_pct = np.percentile(t21_resid, [5, 50, 95], axis=1)    # (3, N_SHOW+1)
t21_filt = filtered(T21, N_ANCHOR)                         # (n_model, n_f) residuals

In [ ]:
C_FG = "k"


def make_figure(path):
    """Colour every model by what it retains, with no class boundaries.

    The three-class version cuts a continuum into bins, which invites
    reading the bins as populations; they are not. Here the colour is the
    retained RMS itself, on a log scale, and the colorbar replaces the
    class legend. Exemplars are dropped -- nothing distinguishes those
    three models once colour carries the quantity directly.
    """
    ret = t21_resid[N_ANCHOR] * 1e3                         # colour quantity [mK]
    norm = LogNorm(vmin=ret[ret > 0].min(), vmax=ret.max())

    # Draw order is randomised, not sorted by retention. Sorting either way
    # puts one end of the colour scale underneath the whole ensemble; a seeded
    # permutation biases neither end, so the rendered density is the
    # ensemble's own. (The opacity ramp below, not this, is what actually
    # rescues the high-retention tail -- order alone cannot, without burying
    # the other end instead.)
    rng = np.random.default_rng(Z_SEED)
    order = rng.permutation(ret.size)

    # Panels (a1)/(a2) draw only the n_draw-model subsample, for legibility;
    # panel (b) below still draws every surviving model. Colour and the
    # colorbar (norm, above) are always keyed to the full ensemble.
    draw_ret = ret[draw_idx]
    draw_order = rng.permutation(draw_idx.size)

    def alpha_for(r, lo, hi):
        """Opacity rising with retained RMS, on the colour scale's own norm."""
        return lo + (hi - lo) * np.clip(np.asarray(norm(r)), 0, 1) ** RAMP_POW

    def segs(x, Y):
        return np.stack([np.broadcast_to(np.asarray(x), Y.shape), Y], axis=-1)

    fig, ax = plt.subplot_mosaic(
        [["a1", "b"], ["a2", "b"]],
        figsize=(7.6, 3.2), layout="constrained",
        gridspec_kw=dict(width_ratios=[1, 1.2]),
    )
    panels = (("a1", T21[draw_idx][draw_order] * 1e3),
              ("a2", t21_filt[draw_idx][draw_order] * 1e3))
    for key, Y in panels:
        lc = LineCollection(
            segs(freqs, Y), cmap=CONT_CMAP, norm=norm, lw=0.4,
            alpha=alpha_for(draw_ret[draw_order], ALL_ALPHA_LO, ALL_ALPHA_HI))
        lc.set_array(draw_ret[draw_order])
        ax[key].add_collection(lc)
        ax[key].set_xlim(freqs[0], freqs[-1])
        ax[key].set_ylim(Y.min() * 1.05, max(Y.max() * 1.05, 0.02 * abs(Y.min())))

    b = ax["b"]
    lc = LineCollection(segs(n_modes, t21_resid[:, order].T), cmap=CONT_CMAP,
                        norm=norm, lw=0.5,
                        alpha=alpha_for(ret[order], ALPHA_LO, ALPHA_HI))
    lc.set_array(ret[order])
    b.add_collection(lc)
    ref = [b.plot(n_modes, fg_resid, color=C_FG, lw=1.5,
                  label="foreground residual")[0]]

    for key, lab, ylab in (("a1", "input", r"$T_{21}$ [mK]"),
                           ("a2", f"after filtering {N_ANCHOR} modes",
                            "Residual [mK]")):
        ax[key].axhline(0, color="0.6", lw=0.6, ls="--", zorder=0)
        ax[key].set_ylabel(ylab, fontsize=8)
        ax[key].grid(alpha=0.2)
        ax[key].tick_params(labelsize=7)
        ax[key].text(0.03, 0.06, lab, transform=ax[key].transAxes, fontsize=7,
                     ha="left", va="bottom")
    ax["a1"].tick_params(labelbottom=False)
    ax["a2"].set_xlabel("Frequency [MHz]", fontsize=8)

    b.axvline(N_ANCHOR, color="0.6", lw=0.8, ls=":", zorder=0)
    b.text(N_ANCHOR - 0.3, 1e0, f"$N = {N_ANCHOR}$", fontsize=7, color="0.35",
           ha="right", va="center")
    b.set_yscale("log")
    b.set_xlim(0, N_SHOW)
    b.set_ylim(1e-5, 3e3)
    b.set_xlabel("Foreground modes filtered", fontsize=8)
    b.set_ylabel("RMS over band [K]", fontsize=8)
    b.grid(True, which="both", ls=":", lw=0.5, alpha=0.6)
    b.tick_params(labelsize=7)
    b.legend(handles=ref, fontsize=6.5, loc="lower left", framealpha=0.92)

    sm = ScalarMappable(norm=norm, cmap=CONT_CMAP)
    cb = fig.colorbar(sm, ax=b, pad=0.015, fraction=0.045)
    cb.set_label(f"21 cm RMS retained at $N = {N_ANCHOR}$ [mK]", fontsize=7.5)
    cb.ax.tick_params(labelsize=6.5)
    cb.solids.set_alpha(1.0)

    fig.savefig(path, bbox_inches="tight", dpi=600)


make_figure("signal_loss.pdf")

In [ ]:
frac_above = (t21_resid > fg_resid[:, None]).mean(axis=1)
print(f"{'N':>3} {'fgnd':>9} {'21cm p50':>9} {'21cm p95':>9} "
      f"{'frac>fgnd':>10}   (mK)")
for N in (6, 8, N_ANCHOR, 10, 12, 15):
    print(f"{N:3d} {fg_resid[N]*1e3:9.3f} "
          f"{t21_pct[1, N]*1e3:9.3f} {t21_pct[2, N]*1e3:9.3f} "
          f"{frac_above[N]:10.2f}")

keep = t21_resid[N_ANCHOR] / t21_resid[0]
print(f"\nAt N = {N_ANCHOR}: median model keeps {np.median(keep)*100:.0f}% of its "
      f"RMS ({t21_pct[1, N_ANCHOR]*1e3:.2f} mK), while the foreground residual is "
      f"{fg_resid[N_ANCHOR]*1e3:.2f} mK.")
print(f"{frac_above[N_ANCHOR]*100:.0f}% of the {t21_resid.shape[1]} models retain "
      f"more signal than the foreground residual.")

# What separates the classes: at matched depth it is trough width, not amplitude.
width = (T21 < T21.min(axis=1, keepdims=True) / 2).sum(axis=1) * (freqs[1] - freqs[0])
depth = -T21.min(axis=1) * 1e3
window = (depth > 80) & (depth < 160)
print()
for k, lab in enumerate(class_labels):
    m, mw = cls == k, (cls == k) & window
    print(f"{lab:>6s} retained: {m.sum():4d} models, median depth "
          f"{np.median(depth[m]):6.1f} mK; at matched depth (80-160 mK) "
          f"n={mw.sum():3d}, median trough width {np.median(width[mw]):3.0f} MHz")